<a href="https://colab.research.google.com/github/princemwaanga/J-Drop/blob/main/RNN_Text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import tensorflow as tf
import numpy as np
import os
import time
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.losses import sparse_categorical_crossentropy

# Load and prepare the dataset
path_to_file = tf.keras.utils.get_file(
    'shakespeare.txt',
    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
)

text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
print(f'Length of text: {len(text)} characters')
print(text[:250])  # Print first 250 characters

# Create character-level vocabulary
vocab = sorted(set(text))
print(f'{len(vocab)} unique characters')

# Create mapping from characters to indices
char2idx = {u:i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

# Convert text to numerical representation
text_as_int = np.array([char2idx[c] for c in text])

# Create training examples
seq_length = 100  # Length of input sequences
examples_per_epoch = len(text) // (seq_length + 1)


# Create the training dataset
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

# Batch and shuffle the data
BATCH_SIZE = 64
BUFFER_SIZE = 10000
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

# Model parameters
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 1024

# Build the model using input_shape instead of batch_input_shape
def build_model(vocab_size, embedding_dim, rnn_units):
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_shape=(None,)),
        LSTM(rnn_units,
             return_sequences=True,
             recurrent_initializer='glorot_uniform'),
        Dense(vocab_size)
    ])
    return model

model = build_model(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units
)

model.summary()

def loss(labels, logits):
    return sparse_categorical_crossentropy(labels, logits, from_logits=True)

model.compile(optimizer='adam', loss=loss)
# Configure checkpoints
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix + ".weights.h5",
    save_weights_only=True
)

# Train the model
EPOCHS = 30
history = model.fit(
    dataset,
    epochs=EPOCHS,
    callbacks=[checkpoint_callback]
)

# Rebuild the model for text generation with batch size 1
#model = build_model(vocab_size, embedding_dim, rnn_units)
#model.load_weights(tf.train.latest_checkpoint(checkpoint_dir))

# Rebuild the model for text generation with batch size 1
model = build_model(vocab_size, embedding_dim, rnn_units)
# Update the checkpoint path to include the .weights.h5 extension
latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)
if latest_checkpoint:
    model.load_weights(latest_checkpoint)
else:
    print("No checkpoint files found in the directory.")

# Text generation function
'''def generate_text(model, start_string, num_generate=1000, temperature=1.0):
    # Vectorize the start string
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    text_generated = []

    # Reset model state for generation
    model.reset_states()
    for i in range(num_generate):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0)

        # Scale predictions by temperature
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(
            predictions,
            num_samples=1
        )[-1, 0].numpy()

        input_eval = tf.expand_dims([predicted_id], 0)
        text_generated.append(idx2char[predicted_id])

    return start_string + ''.join(text_generated)

# Generate some text
print(generate_text(model, start_string="ROMEO: "))'''

# Text generation function
def generate_text(model, start_string, num_generate=1000, temperature=1.0):
    # Vectorize the start string
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    text_generated = []

    # Remove model.reset_states() as it's not needed for stateless models
    # model.reset_states()
    for i in range(num_generate):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0)

        # Scale predictions by temperature
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(
            predictions,
            num_samples=1
        )[-1, 0].numpy()

        input_eval = tf.expand_dims([predicted_id], 0)
        text_generated.append(idx2char[predicted_id])

    return start_string + ''.join(text_generated)

# Generate some text
print(generate_text(model, start_string="ROMEO: "))

Length of text: 1115394 characters
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

65 unique characters


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 256)      │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 1024)     │     5,246,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 65)       │        66,625 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,330,241 (20.33 MB)

 Trainable params: 5,330,241 (20.33 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 1363s 8s/step - loss: 2.8585
Epoch 2/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 1337s 8s/step - loss: 1.8405
Epoch 3/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 1339s 8s/step - loss: 1.5778
Epoch 4/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 1334s 8s/step - loss: 1.4460
Epoch 5/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 1323s 8s/step - loss: 1.3657
Epoch 6/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 1365s 8s/step - loss: 1.3073
Epoch 7/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 1404s 8s/step - loss: 1.2587
Epoch 8/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 1331s 8s/step - loss: 1.2155
Epoch 9/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 1346s 8s/step - loss: 1.1740
Epoch 10/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 1330s 8s/step - loss: 1.1350
Epoch 11/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 1334s 8s/step - loss: 1.0908
Epoch 12/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 1353s 8s/step - loss: 1.0486
Epoch 13/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 1334s 8s/step - loss: 1.0031
Epoch 14/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 1340s 8s/step - loss: 0.9550
Epoch 15/30
172/172 ━━━━━━━━━